In [12]:
from pyspark.sql.functions import col
from pyspark.sql import Row
import scipy.stats as stats

# ================================
# A/B Testing — Classic Cars vs Vintage Cars
# Objective: Is there a significant difference in sales between the two product groups?
# ================================

# Creating groups
group_a = spark.read.table("silver_sales")\
    .filter(col("PRODUCTLINE") == "CLASSIC CARS")\
    .select("SALES").toPandas()["SALES"]

group_b = spark.read.table("silver_sales")\
    .filter(col("PRODUCTLINE") == "VINTAGE CARS")\
    .select("SALES").toPandas()["SALES"]

# T-test implementation
t_stat, p_value = stats.ttest_ind(group_a, group_b)

# Determine the result dynamically.
alpha = 0.05
result_text = "H0 REDDEDİLDİ — Anlamlı fark VAR" \
    if p_value <= alpha else "H0 KABUL EDİLDİ — Anlamlı fark YOK"

# Print the results
print("=" * 50)
print("A/B TEST SONUÇLARI")
print("=" * 50)
print(f"Grup A (Classic Cars) ortalama: ${group_a.mean():.2f}")
print(f"Grup B (Vintage Cars) ortalama: ${group_b.mean():.2f}")
print(f"Fark: ${group_a.mean() - group_b.mean():.2f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Alpha: {alpha}")
print(f"Sonuç: {result_text}")
print("=" * 50)

# Saving our gold table as a result
results = spark.createDataFrame([
    Row(
        test_name="Classic Cars vs Vintage Cars",
        group_a="CLASSIC CARS",
        group_b="VINTAGE CARS",
        mean_a=float(group_a.mean()),
        mean_b=float(group_b.mean()),
        t_statistic=float(t_stat),
        p_value=float(p_value),
        alpha=float(alpha),
        result_text=result_text
    )
])

results.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_ab_test_results")

print("✅ gold_ab_test_results güncellendi!")

StatementMeta(, 99f9d49c-9052-487e-9a9e-6e5c94226e53, 14, Finished, Available, Finished, False)

A/B TEST SONUÇLARI
Grup A (Classic Cars) ortalama: $4053.38
Grup B (Vintage Cars) ortalama: $3135.34
Fark: $918.04
T-statistic: 9.0753
P-value: 0.0000
Alpha: 0.05
Sonuç: H0 REDDEDİLDİ — Anlamlı fark VAR
✅ gold_ab_test_results güncellendi!


* T-statistic: 9.0753 → very large! The two groups are very different from each other.
* P-value: 0.0000 → the probability of it being by chance is almost zero!
* Alpha: 0.05 → our threshold
* H0 REJECTED → the difference is real!